# Vector Search with PGVector

With PG, I can store and retrieve the embedded vectors like chromaDB and on top of that I can write SQL quries and the result is the embedded vectors (It is used in the production)


sourece:: https://www.youtube.com/watch?v=0P54MFyz-mc&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv&index=26


First installation by docker:

docker run -it \
    --name pgvector \
    -e POSTGRES_USER=user \
    -e POSTGRES_PASSWORD=pswd \
    -e POSTGRES_DB=faq \
    -v pgvector_data:/var/lib/postgresql/data \
    -p 5432:5432 \
    pgvector/pgvector:pg17
0

Instead: install it, 
1. go to::: https://www.postgresql.org/download/windows/?utm_source=chatgpt.com 
2. download version 17 for window
3.  data will stored under: C:\Program Files\PostgreSQL\17\data, password=saba, port=5432

### Connect to Postgres and create a DataBase

In [ ]:

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import AutoTokenizer
from langchain_core.documents import Document
from dotenv import load_dotenv
import os
import psycopg
# load the data
from pathlib import Path
from rich import print
import os
import sys
import numpy as np
from sqlitesearch import VectorSearchIndex


parent_directory = os.path.dirname(os.getcwd())
sys.path.append(parent_directory)


load_dotenv()

postgres_admin_password = os.getenv("PostgreSQ_PASSWORD_ADMIN")
postgresuser_username =os.getenv("PostgreSQ_USERNAME")
postgres_user_password =os.getenv("PostgreSQ_PASSWORD")
postgres_port =os.getenv("PostgreSQ_PORT")
postgres_url =os.getenv("PostgreSQ_URL")


# connect to default admin database
# this is an authentication, where I add user, remove user, create a database and remove a database, check existing data base
admin_conn = psycopg.connect(
    f"postgresql://{postgresuser_username}:{postgres_admin_password}@{postgres_url}:{postgres_port}/postgres"
)
admin_conn.autocommit = True

# check if a database is exist
db_name = "faq_postreg"
  
# checking wether database db_name is existed
result = admin_conn.execute(
    "SELECT 1 FROM pg_database WHERE datname = %s",
    (db_name,)
).fetchone()

# create a database if it is not available, and close the connection
if result is None:
    admin_conn.execute(f"CREATE DATABASE {db_name}")
    print(f"Database '{db_name}' created.")
else:
    print(f"Database '{db_name}' already exists.")

admin_conn.close()

Database 'faq_postreg' created.

### Prepare the data

Load the documents and Embed them to get vector


In [ ]:
# 1. load the documents
print('---------------------------------------------------------------')
print('Step 1: load the data')
documents = load_faq_data()
print(f'The number of documents is: {len(documents)}')
# print(documents[0])

print('---------------------------------------------------------------')
print('Step 2: Get the useful data only from each documant')
# extract only the question and answer from each documents
texts = list()
for doc in documents:
    text_per_document = doc['question'] + " " + doc['answer']
    texts.append(text_per_document)
print(f'The number of text documents is: {len(texts)}')
# print(texts[0])

# """
#     NOTE::: the ingestion part::  text_per_document = doc['question'] + " " + doc['answer']
#     can be imporoved, 
#     text = f"""
#     Question:
#     {doc["question"]}

#     Answer:
#     {doc["answer"]}
#     """
#     Then Chroma will return:

#     Document(
#         page_content="""
#         Question:
#         I just discovered the course. Can I still join?

#         Answer:
#         Yes, but if you want a certificate...
#     """


# """
#     )

#     ALSO: we can get the metadata

#     def build_context(self, docs):

#         lines = []

#         for doc in docs:
#             lines.append(f"Section: {doc.metadata['section']}")
#             lines.append(doc.page_content)
#             lines.append("")

#         return "\n".join(lines)
# """
print('---------------------------------------------------------------')
print('Step 3: chunck the data')

# # we can also use:::
# from gitsource import chunk_documents
# chunks = chunk_documents(documents, size=2000, step=1000)

# 3.1. Convert your list of text strings into a list of LangChain Document objects
# This keeps your 1,350 documents separate and un-merged.
docs = [Document(page_content=text) for text in texts]


# Note::: I changed the chunck_size to 5000 so that I get the same numbers of vectors and documents to overcome an error
# 3.2. Setup the Recursive Text Splitter
# It chunks your text smartly, keeping paragraphs/sentences together, and creates overlaps!
# NOTE::: the number of vectors need to be exactly as the number of documents if I use minsearch
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=5000,    # A high number ensures a single Q&A text never gets split up
    # chunk_size=500, # for improvement
    chunk_overlap=0,    # Zero overlap ensures no repeated data, just like your old loop
    # chunk_overlap=50, # for improvement
    # length_function=lambda text: len(tokenizer.encode(text)) # Counting tokens, # for improvement and by default length_function=len
    # separators=[r"Question \d+:", "\n\n", "\n", " "],# for improvement. note custome seperator= separators=["\n===\n", "\n\n", "\n", " "] but since I have only question and answer, I set it differently
    # is_separator_regex=True # for improvement 
)

# split the docs (chuncks)
texts_chuncks = text_splitter.split_documents(docs) # Note: we do not use  text_splitter.split_text(" ".join(texts))  since we have a list of document, each document will be splitted into chuncks, then langchain merges all the chunck
print(f"The number of chunks documents is: {len(texts_chuncks)}")
# print(texts_chuncks[0])

print('---------------------------------------------------------------')
print('Step 4: embed the data')
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-V2", # same model as in datatalk
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True} # this is for normalizing (sumation =1 of a vector)
)

---------------------------------------------------------------

Step 1: load the data

The number of documents is: 1350

---------------------------------------------------------------

Step 2: Get the useful data only from each documant

...

---------------------------------------------------------------

Step 3: chunck the data

The number of chunks documents is: 1350

---------------------------------------------------------------

Step 4: embed the data

### Low Level Ingestion to Postreg DataBase with PG
NOTE: With PG we can do high or low level to insert the embedded vectors to postgress data base

In [6]:
# The embedder needs text not documents, so we need to extract the text
chunk_texts = [
    doc.page_content
    for doc in texts_chuncks
]
print(f'The size of the chunk texts is: {len(chunk_texts)}')

# create embedded vectors
vectors = embeddings.embed_documents(chunk_texts)
print(f'The size of the vectors is: {len(vectors)}')
print(f'Each vector size is: {len(vectors[0])}')

The size of the chunk texts is: 1350

The size of the vectors is: 1350

Each vector size is: 384

In [12]:
type(vectors)

list

In [ ]:
# Check if the current user can connect
admin_conn = psycopg.connect(
    f"postgresql://{postgresuser_username}:{postgres_user_password}@{postgres_url}:{postgres_port}/{db_name}"
)
print(admin_conn.info.user) # for checking assert conn.info.user == "saba"
print(admin_conn.info.dbname) # assert conn.info.dbname == "faq_postreg"


saba

faq_postreg

In [ ]:
# note, This is perfect because PostgreSQL checks for you
# activates pgvector, ships the extension inside, and this turns it on. It adds the vector column type and the similarity search operators.
admin_conn.execute("CREATE EXTENSION IF NOT EXISTS vector")

In [10]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [11]:
# create a table and set its schema, the embedding vector to store the embeddings
admin_conn.execute("""
    CREATE TABLE documents (
        id SERIAL PRIMARY KEY,
        course TEXT,
        section TEXT,
        question TEXT,
        answer TEXT,
        embedding vector(384)
    )
""")

<psycopg.Cursor [COMMAND_OK] [INTRANS] (host=localhost user=saba database=faq_postreg) at 0x2033aa728d0>

In [ ]:
# inseart the documents and vectors inside the database
# def vec_to_str(vector):
#     return "[" + ",".join(str(x) for x in vector) + "]"

# for doc, vec in tqdm(zip(documents, vectors), total=len(documents)):
#     conn.execute(
#         """
#         INSERT INTO documents (course, section, question, answer, embedding)
#         VALUES (%s, %s, %s, %s, %s::vector)
#         """,
#         (doc["course"], doc["section"], doc["question"], doc["answer"],
#          vec_to_str(vec))
#     )

# conn.commit()

# instead of converting the vectors to strin, I will insert them as list
for doc, vec in zip(documents, vectors):
    admin_conn.execute(
        """
        INSERT INTO documents (course, section, question, answer, embedding)
        VALUES (%s, %s, %s, %s, %s)
        """,
        (
            doc["course"],
            doc["section"],
            doc["question"],
            doc["answer"],
            vec
        )
    )

admin_conn.commit()

# NOTE::::  we can do i t also using register_vector
# # note for better injection, we can use:
# from pgvector.psycopg import register_vector

# register_vector(admin_conn)
# admin_conn.execute(
#     "INSERT INTO documents (...) VALUES (%s, %s, %s, %s, %s)",
#     (..., vec)
# )


### High Level Ingestion to Postreg DataBase with PG

In [ ]:
# we can do as Chroma as chroma does::::::::::::::
# chromadb_vector_store = Chroma.from_documents(
#     documents=texts_chuncks,
#     embedding=embeddings,
#     collection_name="datatalk_training",
#     persist_directory=PERSIST_DIRECTORY
# )

# from langchain_postgres import PGVector

# CONNECTION_STRING = (
#     f"postgresql+psycopg://{postgresuser_username}:"
#     f"{postgres_user_password}@{postgres_url}:{postgres_port}/{db_name}"
# )
# vector_store = PGVector(
#     embeddings=embeddings,
#     collection_name="faq",
#     connection=CONNECTION_STRING,
# )

# vector_store.add_documents(documents)

### check if the storing is done correctly

In [27]:
admin_conn.rollback()
# Check row count
row_count = admin_conn.execute("SELECT COUNT(*) FROM documents;").fetchone()
print(f"Total rows in 'documents' table: {row_count[0]}")

if row_count[0] > 0:
    sample = admin_conn.execute("SELECT id, course, question FROM documents LIMIT 3;").fetchall()
    print("\nSample Data:")
    for row in sample:
        print(row)

Total rows in 'documents' table: 1350

Sample Data:

(1, 'data-engineering-zoomcamp', 'Course: When does the course start?')

(2, 'data-engineering-zoomcamp', 'Course: What are the prerequisites for this course?')

(3, 'data-engineering-zoomcamp', 'Course: Can I still join the course after the start date?')

In [ ]:
# Clear any lingering transaction states
admin_conn.rollback()

# Fetch one sample row including the embedding column
sample_vector = admin_conn.execute(
    "SELECT id, question, embedding FROM documents WHERE embedding IS NOT NULL LIMIT 1;"
).fetchone()

if sample_vector:
    v_id, question, vector_str = sample_vector
    
    # pgvector returns vectors as a string format like '[0.123,-0.456,...]' 
    # Let's parse it or inspect its structure
    print(f"✅ Found valid embedding for Row ID: {v_id}")
    print(f"Question text: '{question}'")
    
    # Clean up the string representation to inspect it
    if isinstance(vector_str, str):
        dimensions = len(vector_str.strip("[]").split(","))
        preview = vector_str[:60] + "..."
    else:
        # If pgvector/psycopg extension is fully active, it might already be a list/array
        dimensions = len(vector_str)
        preview = str(vector_str)[:60] + "..."
        
    print(f"Vector Dimensions: {dimensions} (Expected: 384)")
    print(f"Vector Preview: {preview}")
else:
    print("❌ Warning: No rows found with a non-null embedding!")

✅ Found valid embedding for Row ID: 1

Question text: 'Course: When does the course start?'

Vector Dimensions: 384 (Expected: 384)

Vector Preview: [-0.026706211,-0.12245754,0.015944196,0.0060941996,-0.011191...

# create an index so that you fetch fast using HNSW


Right now, when you run your query, PostgreSQL performs a Linear Search (Exact Search). It scans all 1,350 rows, calculates the exact similarity score for every single vector, and sorts them. While perfect for 1,350 rows, if your database grows to 100,000 or millions of rows, this calculation becomes incredibly slow.

So instead of that switche the database to an Approximate Nearest Neighbor (ANN) search using an algorithm called HNSW.


HNSW stands for Hierarchical Navigable Small World.

Instead of looking at every vector one by one, HNSW builds a multi-layered graph data structure out of your vectors, highly resembling a skip-list or a network of "friends of friends."

The Multi-Layer Graph: The top layers have fewer vectors and long-distance connections (like express highways). The bottom layer contains all your vectors with short-distance connections (like local city streets).

The Search Process: The algorithm starts at the top layer, takes huge leaps to find the general neighborhood of your query vector, and then drops down a layer to fine-tune the search. It repeats this until it finds the closest matches in the bottom layer.

In [33]:
admin_conn.execute("""
    CREATE INDEX ON documents 
    USING hnsw (embedding vector_cosine_ops);
""")
admin_conn.commit()
print("HNSW Index built successfully!")

HNSW Index built successfully!

# PHASE 2 (Retriever & RAG with PG)

## Low level

In [31]:
query = "I just discovered the course. Can I still join it?"
query_vector = embeddings.embed_query(query)

# # search from the postreg data base the most similar context
# results = admin_conn.execute(
#     """
#     SELECT course, question, answer,
#            1 - (embedding <=> %s::vector) AS similarity
#     FROM documents
#     ORDER BY embedding <=> %s::vector
#     LIMIT 5
#     """,
#     (query_str, query_str)
# ).fetchall()

# for row in results:
#     print(f"[{row[0]}] {row[1]} (similarity: {row[3]:.4f})")
 # %s when I insert a string of vectors
with admin_conn.transaction():
    results = admin_conn.execute(
        """
        SELECT course, section, question, answer,
               1 - (embedding <=> %s::vector) AS similarity
        FROM documents
        ORDER BY embedding <=> %s::vector
        LIMIT 5;
        """,
        (query_vector, query_vector)
    ).fetchall()

# 3. Print the results
print(f"Successfully retrieved {len(results)} rows:")

for row in results:
    print(f"\n[Score: {row[4]:.4f}] {row[0]} - {row[2]}")

Successfully retrieved 5 rows:

[Score: 0.8365] llm-zoomcamp - I just discovered the course. Can I still join?

[Score: 0.6904] machine-learning-zoomcamp - The course has already started. Can I still join it?

[Score: 0.6043] mlops-zoomcamp - Course - Can I still join the course after the start date?

[Score: 0.5959] data-engineering-zoomcamp - Course: Can I still join the course after the start date?

[Score: 0.5927] data-engineering-zoomcamp - Course: Can I get support if I take the course in the self-paced mode?

# filter by field

In [34]:
# Ensure you are using the actual vector array/list, not the raw text query
query_vector = embeddings.embed_query("I just discovered the course. Can I still join it?")
course_filter = "llm-zoomcamp"

# Clear any previous transaction errors
admin_conn.rollback()

with admin_conn.transaction():
    results = admin_conn.execute(
        """
        SELECT course, section, question, answer,
               1 - (embedding <=> %s::vector) AS similarity
        FROM documents
        WHERE course = %s
        ORDER BY embedding <=> %s::vector
        LIMIT 5;
        """,
        (query_vector, course_filter, query_vector) # <-- Order must match the SQL placeholders perfectly
    ).fetchall()

# Display the filtered results
for row in results:
    print(f"\n[Score: {row[4]:.4f}] {row[0]} ({row[1]})")
    print(f"Q: {row[2]}")

[Score: 0.8365] llm-zoomcamp (General Course-Related Questions)

Q: I just discovered the course. Can I still join?

[Score: 0.5113] llm-zoomcamp (General Course-Related Questions)

Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?

[Score: 0.5090] llm-zoomcamp (General Course-Related Questions)

Q: When will the course be offered next?

[Score: 0.5014] llm-zoomcamp (Module 1: RAG)

Q: Can I run the course locally instead of Codespaces?

[Score: 0.4338] llm-zoomcamp (Module 1: RAG)

Q: OpenAI: Do I have to subscribe and pay for Open AI API for this course?

In [38]:
def pgvector_search(query, course=None, num_results=5):
    """
    Searches documents using pgvector. 
    The 'course' parameter is optional. If provided, results are filtered by course.
    """
    # 1. Generate the query vector
    # (Use embeddings.embed_query(query) or model.encode(query) based on your setup)
    query_vector = embeddings.embed_query(query) 
    
    # 2. Build the base query elements
    base_query = """
        SELECT course, section, question, answer,
               1 - (embedding <=> %(vector)s::vector) AS similarity
        FROM documents
    """
    
    # Initialize parameters dictionary
    params = {
        "vector": query_vector,
        "limit": num_results
    }
    
    # 3. Add the optional WHERE clause dynamically
    where_clause = ""
    if course:
        where_clause = "WHERE course = %(course)s"
        params["course"] = course

    # 4. Combine everything with ORDER BY and LIMIT
    final_query = f"""
        {base_query}
        {where_clause}
        ORDER BY embedding <=> %(vector)s::vector
        LIMIT %(limit)s;
    """
    
    # 5. Execute safely inside a transaction block
    with admin_conn.transaction():
        rows = admin_conn.execute(final_query, params).fetchall()

    # 6. Return structured dictionary results
    return [
        {
            "course": r[0], 
            "section": r[1], 
            "question": r[2], 
            "answer": r[3],
            "similarity": float(r[4])
        }
        for r in rows
    ]

pgvector_search(query)

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'similarity': 0.8365046576852014},
 {'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
  'similarity': 0.6903565114524072},
 {'course': 'mlops-zoomcamp',
  'sec

In [39]:
pgvector_search(query, course='llm-zoomcamp')

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'similarity': 0.8365046576852014},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'similarity': 0.5112918310587516},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will 

## Retreiver high level

In [ ]:
# Note we need to store then we retrive using it and this loke mongoDB

In [ ]:
# w need to update RAG
# 
class RAGPgVector(RAGBase):

    def __init__(self, embedder, conn, **kwargs):
        super().__init__(index=None, **kwargs)
        self.embedder = embedder
        self.conn = conn

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        query_str = vec_to_str(query_vector)

        rows = self.conn.execute(
            """
            SELECT course, section, question, answer
            FROM documents
            WHERE course = %s
            ORDER BY embedding <=> %s::vector
            LIMIT %s
            """,
            (self.course, query_str, num_results)
        ).fetchall()

        return [
            {"course": r[0], "section": r[1], "question": r[2], "answer": r[3]}
            for r in rows
        ]
vector_assistant = RAGPgVector(
    embedder=model,
    conn=conn,
    llm_client=openai_client,
)